# Analysis notebook

Imports from `src/gauge`, reads the database only, and produces every table and figure of the paper. Each table is saved as CSV and each figure as HTML under the results folder (`notebook/`). The text here only says what each item is; interpretation is the owner's (see `docs/RESULTS_SKELETON.md`). Pilot-scale and exploratory.

In [ ]:
import os

import pandas as pd

from gauge.config import load_indicators, load_study
from gauge.db.session import make_engine, session_scope
from gauge.evaluate.frame import load_frame
from gauge.report import figures, tables
from gauge.viewdata import annotator_agreement

try:
    from IPython.display import display
except ImportError:  # run outside Jupyter
    display = print

CONFIG = os.environ.get("GAUGE_CONFIG", "config/study.yaml")
cfg = load_study(CONFIG)
indicators = load_indicators(cfg)
engine = make_engine(cfg.database_url())
OUT = cfg.results_path / "notebook"
OUT.mkdir(parents=True, exist_ok=True)

with session_scope(engine) as s:
    T = tables.all_tables(s)
    T["sample"] = tables.sample_description(s, cfg.study.analysis_fy)
    T["selection_log"] = tables.selection_log(s)
    T["data_quality"] = tables.data_quality(s)
    T["verification"] = tables.verification_summary(s)
    frame = load_frame(s)
    mock = tables.uses_mock_runs(s)
    kappa = {kind: annotator_agreement(s, kind) for kind in ("claim_support", "taxonomy")}

if mock:
    print("MOCK DATA: dry-run numbers exercise the code and say nothing about any real model or company.")


def table(name, title):
    """Show a table, and save its numbers as CSV."""
    df = T[name]
    df.to_csv(OUT / f"{name}.csv", index=False, lineterminator="\n")
    print(title)
    display(df)


def figure(make, data, name):
    """Build `make(data)`, show it and save it as a self-contained HTML file; skip if `data` is empty."""
    if data is None or len(data) == 0:
        print(f"{name}: no data yet")
        return
    fig = make(data)
    fig.write_html(OUT / f"{name}.html", include_plotlyjs="cdn")
    fig.show()


## Data

In [ ]:
table("sample", "Table 1. The study sample by sector, analysis-year reports")

In [ ]:
table("selection_log", "Table 2. Companies excluded, with the reason")

In [ ]:
table("data_quality", "Table 3. Data-quality flags by rule (flags never change data)")

In [ ]:
table("verification", "Table 4. The 10% verification sample against the filings")

## Model behaviour

In [ ]:
table("scorecard", "Table 5. Model scorecard: T2 accuracy, valid share, repeatability")

In [ ]:
card = T["scorecard"]
primary = card[card["role"] == "primary"] if len(card) else card
title = "T2 accuracy against the rule (primary panel), 95% interval"
figure(lambda d: figures.interval_bar(d, "model", "t2_accuracy", "condition", title),
       primary, "fig1_t2_accuracy")

In [ ]:
table("context_effects", "Table 6. Effect of added context (C1 to C2), per model")

In [ ]:
figure(figures.rating_shift_figure, frame[frame["valid"]], "fig2_rating_shift")

In [ ]:
figure(figures.transition_stack, T["context_effects"], "fig3_transitions")

## Agreement

In [ ]:
table("agreement_between_models", "Table 7. Agreement between models (primary panel), bootstrap 95% intervals")

In [ ]:
alphas = (("t2_alpha_nominal", "fig4_alpha_t2"), ("t3_alpha_ordinal", "fig5_alpha_t3"))
for column, name in alphas:
    between = T["agreement_between_models"]
    figure(lambda d, c=column: figures.interval_bar(d, "condition", c, None, c), between, name)

In [ ]:
table("attribute_agreement", "Table 8. Attribute agreement analysis against the T2 rule")

## Hypotheses

In [ ]:
table("hypotheses", "Table 9. Hypotheses: metric, test, falsification condition and the mechanical outcome")

In [ ]:
table("hypothesis_tests", "Table 10. Test results behind the hypotheses")

## Controlled baseline, position, grounding, reporting

In [ ]:
table("baseline_vs_panel", "Table 11. Primary panel vs controlled baseline (descriptive)")

In [ ]:
table("position_effect", "Table 12. Position effect (descriptive)")

In [ ]:
figure(figures.tercile_figure, T["position_effect"], "fig6_position")

In [ ]:
table("grounding", "Table 13. Grounding of numeric claims and automatic error taxonomy")

In [ ]:
print("Annotator agreement (Cohen's kappa):")
display(pd.DataFrame(kappa).T)

In [ ]:
table("parse_failure_rates", "Table 14. Parse-failure rates per model")

In [ ]:
table("run_counts", "Table 15. Runs logged")